# Geometry metrics laboratory
Compare finite point sets, ordered polylines and enclosing circles. All coordinates are illustrative planar units.


In [ ]:
from pathlib import Path
import sys
for parent in (Path.cwd(), *Path.cwd().parents):
    package = parent / "implementations" / "python" / "geometry-metrics"
    if package.is_dir():
        sys.path.insert(0, str(package)); break
else:
    raise RuntimeError("Could not locate geometry-metrics package")
from geometry_metrics import hausdorff_distance, directed_hausdorff_distance, discrete_frechet_distance, smallest_enclosing_circle
import matplotlib.pyplot as plt
import numpy as np


## 1. Hausdorff versus discrete Fréchet
Hausdorff ignores sequence order; discrete Fréchet preserves it.


In [ ]:
route_a = [(0,0),(1,0),(2,0),(3,0),(4,0)]
route_b = [(0,0),(1,0.2),(2,0.4),(3,0.2),(4,0)]
print("directed A to B", directed_hausdorff_distance(route_a, route_b))
print("symmetric Hausdorff", hausdorff_distance(route_a, route_b))
print("discrete Frechet", discrete_frechet_distance(route_a, route_b))
print("Frechet after reversing B", discrete_frechet_distance(route_a, route_b[::-1]))


## 3. Rotating-calipers diameter
The farthest pair of a finite point set lies on its convex hull. Compare the rotating-calipers result with brute force.


In [ ]:
from geometry_metrics import convex_polygon_diameter
cloud = [(0,0),(4,0),(4,1),(0,1),(2,0.3),(1,0.7)]
a, b, diameter = convex_polygon_diameter(cloud)
brute = max(np.linalg.norm(np.subtract(p,q)) for i,p in enumerate(cloud) for q in cloud[i+1:])
print("diameter endpoints", a, b, "distance", diameter)
assert np.isclose(diameter, brute)


## 2. Smallest enclosing circle
The minimum circle is determined by at most three boundary points.


In [ ]:
points = [(0,0),(4,0),(1,2),(2,1),(3,3)]
cx, cy, radius = smallest_enclosing_circle(points, seed=7)
print({"center": (cx,cy), "radius": radius})
assert all(np.hypot(x-cx,y-cy) <= radius + 1e-9 for x,y in points)


In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ax.plot(*zip(*route_a), marker='o', label='route A')
ax.plot(*zip(*route_b), marker='s', label='route B')
circle = plt.Circle((cx,cy), radius, fill=False, label='smallest enclosing circle')
ax.add_patch(circle)
ax.scatter(*zip(*points), marker='x', label='point set')
ax.set_aspect('equal'); ax.grid(True); ax.legend()


## 4. Pole of inaccessibility (Polylabel)
Find a high-clearance label point inside a concave polygon with a hole. Lower precision means finer subdivision.


In [ ]:
from geometry_metrics import polylabel
outer = [(0,0),(10,0),(10,10),(6,10),(6,4),(4,4),(4,10),(0,10)]
hole = [(1,1),(2,1),(2,2),(1,2)]
label_x, label_y, clearance, cells = polylabel(outer, holes=[hole], precision=0.02)
print({"label_point": (label_x,label_y), "boundary_clearance": clearance, "cells_examined": cells})
assert clearance > 0


## Interpretation and limitations
Hausdorff compares sampled sets, discrete Fréchet compares ordered vertex sequences, and continuous Fréchet compares the traversed curves. The enclosing circle uses randomized Welzl recursion and floating-point predicates; it is not a certified exact-arithmetic or geodesic method.
